# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 10 · Complete RAG Pipeline Evaluation

Evaluating the complete RAG pipeline is essential to determine whether the system can generate accurate, relevant answers grounded in the information retrieved from the source repository. Unlike retrieval-only evaluation, which focuses on the quality of the retrieved documents, end-to-end evaluation considers both the retrieved context and the final answer generated by the LLM.

This notebook evaluates the developed RAG system using **Ragas and an LLM-as-a-Judge approach**. Two complementary metrics are considered **Faithfulness** and  **Answer Relevancy**.

The evaluation uses a dataset of **20 user queries**, together with the corresponding retrieved contexts and LLM-generated answers. The dataset is stored in `evaluation/complete_rag/evaluation_dataset.json`. Its construction process is reproducible and documented in `evaluation/complete_rag/build_evaluation_dataset.ipynb`.

The notebook introduces the evaluation metrics, applies them to the dataset, and analyses the results to assess the strengths and weaknesses of the developed RAG system. It also discusses the **limitations of LLM-as-a-Judge evaluation**, including the influence of the judge model, the limited size of the evaluation dataset, and the fact that automated scores do not guarantee factual correctness.

### Notebook Objectives

* Explain how Ragas can be used to evaluate a complete RAG pipeline using an LLM-as-a-Judge approach.
* Introduce and apply the Faithfulness and Answer Relevancy metrics.
* Analyse the evaluation results obtained using the [`spark-active-learning`](https://github.com/davidfernxndez/spark-active-learning) repository as the reference knowledge source.
* Discuss the main limitations of the evaluation methodology and the conclusions that can reasonably be drawn from the results.

## Imports and Configuration

In [ ]:
import json
import numpy as np
from pathlib import Path
import asyncio
import json
from IPython.display import Markdown, display

from github_rag.evaluator import RAGEvaluator

## 1. Ragas and an LLM-as-a-Judge Approach

**Ragas** is an open-source framework designed to evaluate Retrieval-Augmented Generation (RAG) systems. It provides a set of metrics to assess different aspects of RAG performance, including the quality of retrieved context and the relevance and faithfulness of generated answers. These metrics help identify weaknesses in the pipeline and guide further improvements.

One of the evaluation approaches supported by Ragas is **LLM-as-a-Judge**, in which a Large Language Model (LLM) acts as an automated evaluator. Instead of relying exclusively on manually annotated answers or exact-match comparisons, the judge model analyses the user query, the retrieved context, and the generated answer to assess its quality according to predefined criteria.

## 2. Evaluation Metrics

This evaluation uses two metrics provided by Ragas: **Faithfulness** and **Answer Relevancy**. Both metrics rely on an LLM-as-a-Judge to assess the quality of the generated answer, but they focus on different aspects of the RAG pipeline.

### 2.1 Faithfulness

**Faithfulness** measures whether the claims made in the generated answer are supported by the retrieved context. Its purpose is to identify answers that introduce unsupported information, even when that information sounds plausible.

Ragas evaluates faithfulness by decomposing the generated answer into individual claims and checking whether each claim can be inferred from the retrieved context. The score is calculated as the proportion of claims supported by the context:

$$
\text{Faithfulness} =
\frac{\text{Supported claims}}{\text{Total claims}}
$$

A score closer to **1** indicates that most or all claims are supported by the retrieved context, whereas a score closer to **0** indicates that many claims lack contextual support.

This metric evaluates the consistency between the answer and the retrieved information; it does not establish whether the retrieved context itself is factually correct or whether the answer provides all the information required.

### 2.2 Answer Relevancy

**Answer Relevancy** measures how relevant the generated answer is to the user's question. It focuses on whether the response addresses the information requested, rather than whether its claims are supported by the retrieved context.

Ragas estimates answer relevancy by using an LLM to generate potential questions based on the answer. These generated questions are then compared with the original user query using semantic similarity. The final score is calculated as the average similarity between the original query and the generated questions.

A score closer to **1** indicates that the answer is highly relevant to the user's question, while a lower score suggests that the response may be incomplete, off-topic, or focused on unrelated information.

Unlike Faithfulness, Answer Relevancy does not directly verify the answer against the retrieved context. Therefore, a response can be relevant to the question while still containing unsupported or incorrect claims.

## 3. Evaluation Methodology Limitations

The evaluation results should be interpreted in light of several limitations related to both the **LLM-as-a-Judge methodology** and the design of the evaluation dataset.

### 3.1 Limitations of LLM-as-a-Judge

Although LLM-as-a-Judge provides a practical way to evaluate natural-language responses automatically, it introduces several potential sources of bias and uncertainty:

* **Judge model dependence:** Evaluation scores depend on the capabilities and behaviour of the LLM used as the judge. Different models may assign different scores to the same answer.
* **Evaluation subjectivity:** Some quality criteria require semantic interpretation, which may lead to inconsistent or imperfect judgements.
* **Limited factual verification:** Faithfulness measures whether claims are supported by the retrieved context, not whether that context is factually correct. Similarly, Answer Relevancy measures alignment with the question rather than factual accuracy.

Consequently, these metrics should be considered automated estimates of answer quality rather than definitive measures of correctness. Manual inspection of representative examples remains valuable for validating the results and identifying errors that the metrics may overlook.

### 3.2 Limitations of the Evaluation Dataset

The evaluation dataset contains only **20 user queries** associated with a single GitHub repository. This limited scope is primarily motivated by the API usage constraints associated with both answer generation and LLM-based evaluation.

This introduces several limitations:

* **Limited sample size:** Twenty queries may not adequately represent the full range of possible user questions, query complexities, and retrieval challenges.
* **Repository-specific evaluation:** The results reflect the behaviour of the system when applied to a particular repository and cannot necessarily be generalised to repositories with different structures, programming languages, or documentation styles.
* **Limited statistical reliability:** A small dataset makes the results more sensitive to individual queries, so the observed scores may not accurately represent performance across a broader range of use cases.

### 3.3 Interpretation of the Results

Given these limitations, the results should be interpreted as an **initial assessment of the developed RAG system**, rather than a comprehensive or statistically conclusive evaluation.

The scores provide useful indications of whether the system generates answers that are relevant to user queries and grounded in the retrieved context. They can also help identify potential weaknesses and guide future improvements.

However, the results do not establish that the system will perform equally well across all queries or GitHub repositories. A more robust evaluation would require a larger and more diverse dataset, tests across multiple repositories, and manual validation of a representative subset of the generated answers.

Despite these limitations, this evaluation provides a useful first step towards systematically assessing the complete RAG pipeline and demonstrates how LLM-as-a-Judge can be incorporated into the development process.

## 4. Complete RAG evaluation procedure

In [ ]:
# Load the evaluation dataset
dataset_path = Path(
    "../evaluation/complete_rag/evaluation_dataset.json"
)

with dataset_path.open("r", encoding="utf-8") as file:
    evaluation_dataset = json.load(file)

print(f"Loaded {len(evaluation_dataset)} queries.")

Loaded 20 queries.


In [ ]:
# Load the RAGEvaluator class that implements Ragas framework
evaluator = RAGEvaluator()

The individual evaluation results for each query are presented below, allowing for a manual inspection of the scores and a qualitative assessment of the system's performance.

In [10]:
async def evaluate_with_retry(
    evaluation_function,
    *,
    max_attempts: int = 3,
    initial_wait: int = 20,
    **kwargs,
) -> float:
    """Evaluate a RAG metric with retries for rate-limit errors.

    Args:
        evaluation_function: Asynchronous evaluation function.
        max_attempts: Maximum number of attempts.
        initial_wait: Initial retry delay in seconds.
        **kwargs: Arguments passed to the evaluation function.

    Returns:
        Evaluation score between 0 and 1.

    Raises:
        Exception: If evaluation fails after the allowed attempts
            or a non-rate-limit error occurs.
    """
    for attempt in range(max_attempts):
        try:
            return await evaluation_function(**kwargs)

        except Exception as exc:
            error_message = str(exc).lower()

            is_rate_limit = (
                "429" in error_message
                or "rate_limit_exceeded" in error_message
                or "tokens per minute" in error_message
            )

            if not is_rate_limit or attempt == max_attempts - 1:
                raise

            wait_seconds = initial_wait * (2 ** attempt)

            print(
                f"Rate limit reached. Retrying in {wait_seconds}s "
                f"(attempt {attempt + 2}/{max_attempts})."
            )

            await asyncio.sleep(wait_seconds)

    raise RuntimeError("Evaluation failed unexpectedly.")


# Evaluation loop
results = []
results_path = Path("ragas_evaluation_results.json")

for index, sample in enumerate(evaluation_dataset, start=1):
    user_input = sample["user_input"]
    retrieved_contexts = sample["retrieved_contexts"]
    answer = sample["response"]

    print(f"\n{'=' * 80}")
    print(f"Query {index}/{len(evaluation_dataset)}")
    print(f"{'=' * 80}")
    print(user_input)
    print()

    display(Markdown(answer))

    result = {
        "user_input": user_input,
        "answer": answer,
        "faithfulness": None,
        "answer_relevancy": None,
        "errors": {},
    }

    # Evaluate Faithfulness
    try:
        result["faithfulness"] = await evaluate_with_retry(
            evaluator.evaluate_faithfulness,
            user_input=user_input,
            response=answer,
            retrieved_contexts=retrieved_contexts,
        )
        print(f"Faithfulness: {result['faithfulness']:.4f}")

    except Exception as exc:
        result["errors"]["faithfulness"] = str(exc)
        print(f"Faithfulness failed: {exc}")

    # Evaluate Answer Relevancy
    try:
        result["answer_relevancy"] = await evaluate_with_retry(
            evaluator.evaluate_answer_relevancy,
            user_input=user_input,
            response=answer,
        )
        print(f"Answer relevancy: {result['answer_relevancy']:.4f}")

    except Exception as exc:
        result["errors"]["answer_relevancy"] = str(exc)
        print(f"Answer relevancy failed: {exc}")

    # Store the result, even if one or both metrics failed
    results.append(result)

    # Save progress after every query
    with results_path.open("w", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False, indent=2)

    print(f"Progress saved: {index}/{len(evaluation_dataset)}")

    # Wait before evaluating the next query
    if index < len(evaluation_dataset):
        print("Waiting 60 seconds...")
        await asyncio.sleep(60)

print(f"\nEvaluation finished. Results saved to: {results_path}")


Query 1/20
What clustering method is used for diversity?



The diversity‑based selection is performed with **Bisecting K‑Means** clustering.  
The `diversity_k_means_selection` routine fits a BisectingKMeans model, assigns each candidate to a cluster, and then picks the instance closest to each cluster’s centroid.  

Sources  
- [AL_methods.py](https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py)  
- [main.ipynb](https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb)

Faithfulness: 0.7500
Answer relevancy: 0.7849
Progress saved: 1/20
Waiting 60 seconds...

Query 2/20
Why isn't the dataset included in the repository?



The HIGGS dataset is intentionally omitted from the repository because it is very large. The project’s `data_info.md` explicitly states that the HIGGS datasets are not included in the repository due to their size, and users are expected to generate or download them locally using the provided data‑preparation script.  

**Source:** [data_info.md](https://github.com/davidfernxndez/spark-active-learning/blob/main/data/data_info.md)

Faithfulness: 1.0000
Answer relevancy: 0.7529
Progress saved: 2/20
Waiting 60 seconds...

Query 3/20
What is the classifier trained and evaluated in the solution?



...

Faithfulness failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4fwrb5qe2sa5c9k8m4k2ps2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197856, Requested 2830. Please try again in 4m56.352s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Rate limit reached. Retrying in 20s (attempt 2/3).


API call failed on attempt 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4fwrb5qe2sa5c9k8m4k2ps2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197810, Requested 3030. Please try again in 6m2.88s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4fwrb5qe2sa5c9k8m4k2ps2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197810, Requested 3030. Please try again in 6m2.88s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Rate limit reached. Retrying in 40s (attempt 3/3).


API call failed on attempt 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4fwrb5qe2sa5c9k8m4k2ps2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197717, Requested 3030. Please try again in 5m22.704s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4fwrb5qe2sa5c9k8m4k2ps2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197717, Requested 3030. Please try again in 5m22.704s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Answer relevancy failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4fwrb5qe2sa5c9k8m4k2ps2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197717, Requested 3030. Please try again in 5m22.704s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Progress saved: 20/20

Evaluation finished. Results saved to: ragas_evaluation_results.json


## 5. Results discussion

This section presents the average results obtained, along with their standard deviation, and provides a discussion of the system's performance based on these metrics.

In [12]:
# Load evaluation results
with open("ragas_evaluation_results.json", "r", encoding="utf-8") as file:
    results = json.load(file)

# Extract valid scores, ignoring failed evaluations
faithfulness_scores = [
    r["faithfulness"]
    for r in results
    if r.get("faithfulness") is not None
]

answer_relevancy_scores = [
    r["answer_relevancy"]
    for r in results
    if r.get("answer_relevancy") is not None
]

# Calculate mean and standard deviation
def calculate_statistics(scores):
    n = len(scores)

    if n == 0:
        return 0.0, 0.0

    mean = np.mean(scores)
    std = np.std(scores, ddof=1) if n > 1 else 0.0

    return mean, std


faith_mean, faith_std = calculate_statistics(faithfulness_scores)
rel_mean, rel_std = calculate_statistics(answer_relevancy_scores)

# Display summary
summary_md = f"""
### 📊 LLM-as-a-Judge Evaluation Results (N = {len(results)})

| Metric | Valid scores | Score ($\\mu \\pm \\sigma$) |
| :--- | :---: | :---: |
| **Faithfulness** | {len(faithfulness_scores)}/{len(results)} | `{faith_mean:.3f} ± {faith_std:.3f}` |
| **Answer Relevancy** | {len(answer_relevancy_scores)}/{len(results)} | `{rel_mean:.3f} ± {rel_std:.3f}` |
"""

display(Markdown(summary_md))


### 📊 LLM-as-a-Judge Evaluation Results (N = 20)

| Metric | Valid scores | Score ($\mu \pm \sigma$) |
| :--- | :---: | :---: |
| **Faithfulness** | 19/20 | `0.856 ± 0.215` |
| **Answer Relevancy** | 18/20 | `0.700 ± 0.207` |


The evaluation results show a mean **Faithfulness of 0.856 ± 0.215** and a mean **Answer Relevancy of 0.700 ± 0.207**.

The Faithfulness score suggests that the generated answers are generally well grounded in the retrieved context, although the relatively high standard deviation indicates considerable variability across queries. Some answers may contain unsupported claims, highlighting the importance of examining individual results rather than relying exclusively on the average score.

Answer Relevancy is lower, with a mean of 0.700, suggesting that the generated answers do not always align closely with the user's questions according to the evaluation model. However, this result should be interpreted cautiously. **Manual inspection of the individual query results reveals limitations of the LLM-as-a-Judge approach**, particularly when dealing with more general or open-ended questions. In some cases, the generated answer addresses the question correctly and provides useful information, yet receives a relatively low Answer Relevancy score. This may occur because the judge model does not fully recognise the semantic relationship between a broad question and an answer that provides a more detailed explanation.

Overall, these results indicate that the RAG system generally produces answers grounded in the retrieved context, while answer relevance appears to be more challenging to assess consistently. The discrepancies observed during manual inspection also demonstrate that automated evaluation metrics should not be interpreted in isolation. **Combining quantitative metrics with qualitative inspection provides a more reliable assessment of RAG performance**, especially when the evaluation dataset is small and the judge model may misinterpret some responses.